# PyTorch 工業影像異常偵測 Demo

這份 Notebook 會在 Google Colab 依序執行三種 CPU Demo：卷積自編碼器、逐像素參考法、局部區塊 KNN，並顯示合成正常／瑕疵範例的結果。Notebook 會自動從公開 GitHub 儲存庫載入程式碼。

> 合成資料只驗證程式流程，不代表真實產品檢測效能。正式使用前須以實際資料驗證與校準。

專案推送到 GitHub 後，可在 Colab 開啟此 Notebook，依序執行儲存格。Notebook 會在 Colab 產生 Demo 結果。

In [ ]:
import numpy as np
import PIL
import torch

print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
print(f"NumPy: {np.__version__} | Pillow: {PIL.__version__}")

In [ ]:
import subprocess
from pathlib import Path

repo_url = "https://github.com/goalanwork-rgb/pytorch_demo.git"
project_root = Path("/content/pytorch_demo")
if not (project_root / "anomaly_detector.py").is_file():
    subprocess.run(["git", "clone", "--depth", "1", repo_url, str(project_root)], check=True)
for filename in ("pixel_reference_detector.py", "patch_knn_detector.py"):
    if not (project_root / filename).is_file():
        raise FileNotFoundError(f"GitHub 專案中找不到 {filename}。")
print(f"專案已載入：{project_root}")

In [ ]:
import subprocess
import sys

demos = [
    ("anomaly_detector.py", "autoencoder", ["--epochs", "2", "--image-size", "64", "--device", "cpu"]),
    ("pixel_reference_detector.py", "pixel_reference", ["--image-size", "64"]),
    ("patch_knn_detector.py", "patch_knn", ["--image-size", "64"]),
]
for script, name, options in demos:
    output_dir = project_root / "artifacts" / f"colab_{name}"
    command = [sys.executable, script, "demo", "--output-dir", str(output_dir), *options]
    print(f"\n執行 {script} ...")
    subprocess.run(command, cwd=project_root, check=True)

In [ ]:
from IPython.display import display
from PIL import Image

for _, name, _ in demos:
    result_dir = project_root / "artifacts" / f"colab_{name}" / "demo_results"
    print(f"\n{name}")
    for suffix in ("clean", "defect"):
        overlay = result_dir / f"demo_{suffix}_overlay.png"
        if overlay.is_file():
            print(f"{suffix} overlay")
            display(Image.open(overlay))
        result_json = result_dir / f"demo_{suffix}_result.json"
        if result_json.is_file():
            print(result_json.read_text(encoding="utf-8"))